# Similarity Search: Euclidean Distance vs. Cosine Similarity & Dot Product
**Author:** Ali ElSharkawi  
**Date:** June 13, 2026

In this notebook, we translate raw vector geometry into production Python code. We map out how algorithms see relationships using straight-line physical distance vs. angular projections.

In [1]:
import numpy as np
from scipy.spatial.distance import euclidean
from sklearn.metrics.pairwise import cosine_similarity

print("Environment configured successfully!")

Environment configured successfully!


## 1. Euclidean Distance ($L_2$ Distance)

### 🔹 Conceptual Definition
Euclidean distance measures the similarity between multiple vectors by calculating the strength of the **difference-vector** ($\vec{v}_1 - \vec{v}_2$). The lower the magnitude of this difference-vector, the more geometrically similar the two points are.

### 🔹 Mathematical Formula
For generalized $n$-dimensional vectors $A$ and $B$:
$$A = (a_1, a_2, \dots, a_n)$$
$$B = (b_1, b_2, \dots, b_n)$$
$$d(A,B) = \sqrt{\sum (A_i - B_i)^2}$$

### 🔹 Core Rule
Use Euclidean distance when **actual numbers and their absolute differences carry critical meaning**.

In [2]:
# --- PRACTICAL EXAMPLES FOR EUCLIDEAN DISTANCE ---

# Example A: Finding similar properties based on price & size
target_mansion  = np.array([5, 5, 45, 1200]) 
similar_mansion = np.array([5, 4, 42, 1150]) 
small_condo     = np.array([1, 1,  8,  150])  

dist_mansion = euclidean(target_mansion, similar_mansion)
dist_condo   = euclidean(target_mansion, small_condo)

print("--- Real Estate Match Results ---")
print(f"Distance to Similar Mansion: {dist_mansion:.2f} -> Interpretation: [CLOSE MATCH] Points are highly clustered.")
print(f"Distance to Small Condo:     {dist_condo:.2f} -> Interpretation: [NOT SIMILAR] Massive structural gap in scale.\n")

# Example B: K-Means Clustering (Grouping points based on sets of characteristics)
customers = np.array([
    [21, 45],   # Young Professional A
    [23, 48],   # Young Professional B
    [54, 320],  # Executive A
    [52, 290]   # Executive B
])

print("--- Customer Segment Analysis ---")
peer_gap = np.linalg.norm(customers[0] - customers[1])
tier_gap = np.linalg.norm(customers[0] - customers[2])

print(f"Distance between peers in same bracket:  {peer_gap:.2f}  -> Interpretation: [VERY CLOSE] Belongs in the same cluster.")
print(f"Distance crossing into executive tier:   {tier_gap:.2f} -> Interpretation: [NOT CLOSE AT ALL] Belongs in separate clusters.")

--- Real Estate Match Results ---
Distance to Similar Mansion: 50.10 -> Interpretation: [CLOSE MATCH] Points are highly clustered.
Distance to Small Condo:     1050.67 -> Interpretation: [NOT SIMILAR] Massive structural gap in scale.

--- Customer Segment Analysis ---
Distance between peers in same bracket:  3.61  -> Interpretation: [VERY CLOSE] Belongs in the same cluster.
Distance crossing into executive tier:   276.97 -> Interpretation: [NOT CLOSE AT ALL] Belongs in separate clusters.


## 2. Dot Product & Cosine Similarity

### 🔹 The Dot Product Definition
The Dot Product measures the similarity between multiple vectors by calculating the **projection of vector $\vec{v}$ onto vector $\vec{u}$**. 
$$\vec{v} \cdot \vec{u} = \|\vec{v}\| \|\vec{u}\| \cos\theta$$

### 🔹 Cosine Similarity Definition
What is the magnitude of $\vec{v}$ in the direction of $\vec{u}$? The "shadow" cast by vector $\vec{v}$ onto $\vec{u}$ is defined by $\|\vec{v}\|\cos\theta$. 

Cosine similarity isolates this relationship by **only caring about the angles between the vectors**, ignoring total magnitude. It is used when relationships between features are crucial regardless of the vector size.
$$\cos\theta = \frac{\vec{v} \cdot \vec{u}}{\|\vec{v}\| \|\vec{u}\|}$$

### 🔹 Production Rule
* **Cosine Similarity:** Range is strictly $[-1, 1]$. Used heavily in Text Classification, NLP, LLM Embeddings, Semantic Search, and Collaborative Filtering.
* **Dot Product:** Used when **speed is crucial**. In production, we pre-normalize vectors to a length of 1 ($\|\vec{v}\|=1$). This matches Cosine Similarity exactly while executing with extreme performance.

In [5]:
# --- PRACTICAL EXAMPLES FOR ANGULAR SIMILARITY ---
user_profile = np.array([2, 0])     
short_tweet  = np.array([1, 0])     
massive_blog = np.array([50, 0])    
cooking_recipe = np.array([0, 5])   

print("--- Vector Matching Analysis ---")

# 1. Same direction, short scale
cos_short = cosine_similarity([user_profile], [short_tweet])[0][0]
print(f"📊 User Profile vs Short Tweet (Same direction):")
print(f" -> Cosine Sim: {cos_short:.4f} -> Interpretation: [PERFECT MATCH (1.0)] Identical intent, small scale gap.")

# 2. Same direction, massive scale discrepancy
cos_blog = cosine_similarity([user_profile], [massive_blog])[0][0]
print(f"📊 User Profile vs Massive Blog (Same direction, massive scale):")
print(f" -> Cosine Sim: {cos_blog:.4f} -> Interpretation: [PERFECT MATCH (1.0)] Identical intent! Scale completely ignored.")

# 3. Orthogonal vector
cos_cook = cosine_similarity([user_profile], [cooking_recipe])[0][0]
print(f"📊 User Profile vs Cooking Recipe (Orthogonal / 90 degrees):")
print(f" -> Cosine Sim: {cos_cook:.4f} -> Interpretation: [NOT SIMILAR (0.0)] No semantic overlap whatsoever.")

--- Vector Matching Analysis ---
📊 User Profile vs Short Tweet (Same direction):
 -> Cosine Sim: 1.0000 -> Interpretation: [PERFECT MATCH (1.0)] Identical intent, small scale gap.
📊 User Profile vs Massive Blog (Same direction, massive scale):
 -> Cosine Sim: 1.0000 -> Interpretation: [PERFECT MATCH (1.0)] Identical intent! Scale completely ignored.
📊 User Profile vs Cooking Recipe (Orthogonal / 90 degrees):
 -> Cosine Sim: 0.0000 -> Interpretation: [NOT SIMILAR (0.0)] No semantic overlap whatsoever.


## 3. Quick Interpretation Matrix

Let's programmatically reproduce the exact truth table drawn in the handwritten notes to observe the boundaries of our metrics across different vector structural relationships.

In [4]:
# Define structural vector pairings matching the notes' table
vec_A = np.array([3, 0])

# 1. Identical vectors
identical = np.array([3, 0])

# 2. Same direction, different lengths
scaled = np.array([12, 0])

# 3. Orthogonal (90 degrees)
orthogonal = np.array([0, 3])

# 4. Opposite direction
opposite = np.array([-3, 0])

relationships = {
    "Identical Vectors": identical,
    "Same Direction, Diff Lengths": scaled,
    "Orthogonal (90° gap)": orthogonal,
    "Opposite Direction": opposite
}

print(f"{'Vector Relationship':<30} | {'Dot Product':<12} | {'Cosine Similarity':<20} | {'Euclidean Distance'}")
print("-" * 90)

for name, vec_B in relationships.items():
    dot_p = np.dot(vec_A, vec_B)
    cos_s = cosine_similarity([vec_A], [vec_B])[0][0]
    eucl_d = euclidean(vec_A, vec_B)
    
    # Custom display strings to align cleanly with your handwritten interpretation notes
    print(f"{name:<30} | {dot_p:<12.1f} | {cos_s:<20.4f} | {eucl_d:.4f}")

Vector Relationship            | Dot Product  | Cosine Similarity    | Euclidean Distance
------------------------------------------------------------------------------------------
Identical Vectors              | 9.0          | 1.0000               | 0.0000
Same Direction, Diff Lengths   | 36.0         | 1.0000               | 9.0000
Orthogonal (90° gap)           | 0.0          | 0.0000               | 4.2426
Opposite Direction             | -9.0         | -1.0000              | 6.0000
